# Omni multi-vector retrieval trên YouCook2 (UEMR P3 + P4, chưa có Athena)

Encode ~500 video YouCook2 bằng checkpoint Omni fine-tune (`youcookii_ft_uemr/best`, cùng checkpoint `Omni_inference_uemr.ipynb` đã chấm), truy vấn tương tác, chấm nhanh R@1/5/10, MedR text→video cho `global`, `gt`, `uni_M_gt` (+ `event_single`).

**Khác `Omni_inference_uemr.ipynb` ở đâu:** notebook đó chấm benchmark bằng `omniretriever.cli extract` (PyAV, center-crop vuông, sàn 336 px, `transformers==4.57.3`). Ở đây vector phải **trùng với vector mà `eval_youcookii.py` đã đo** khi chọn checkpoint (`best_val.json`), nên encode bằng chính pipeline đó (`LazySupervisedDataset`: decord, 8 frame, 50176 px giữ tỉ lệ, audio cắt từ mp4 rồi center-crop 8 s, `<|AUDIO|>` nhân đôi) và cài **đúng môi trường của notebook training** (`training/requirements.txt`, `transformers==4.51.3`). Text = `caption<|im_end|>` → hidden state lớp cuối ở token cuối (nhánh label của eval), không phải `OmniRetriever.encode_text`. Không cần vá M-RoPE (pipeline eval dùng model vendored trong `training/qwenvl`).

Cache khử trùng lặp theo `(video, ts, te)`; thêm partition mới (proposals của Athena) chỉ encode đoạn chưa có. Mọi cell chạy lại được an toàn.

**Thứ tự chạy**
1. Cell 1–5: setup (code, thư viện, WAVE-7B, checkpoint + kiểm tra fusion head, cấu hình).
2. Cell 6–8: unit test, dry-run, test collect/evaluate bằng npz giả. Không cần GPU, không nạp model.
3. Cell 9–16 với `SMOKE = True` (5 video, encode thật).
4. Smoke ổn → đặt `SMOKE = False` ở cell 5, chạy lại cell 5 rồi 9–16 (embedding của smoke được dùng lại).
5. Cell 17–18: giải phóng bộ nhớ, query tương tác.

**Runtime:** A100.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Lấy code và cài thư viện

* Omni-fix nhánh `chien` → `/content/Omni-fix` (như `Omni_inference_uemr.ipynb`), không sửa gì trong đó.
* Fushion: `CODE_SOURCE = 'git'` clone `chien24/Fushion_UEMR` (pull nếu đã có); `'drive'` copy từ `FUSHION_DRIVE`.
* Thư viện: **như notebook training** (`requirements.txt` rồi `training/requirements.txt`, ra `transformers==4.51.3`) vì đó là môi trường `eval_youcookii.py` đã chạy ra `best_val.json`. Không dùng pin `transformers==4.57.3` của notebook inference (pin đó cho đường CLI). Sau khi cài lần đầu, nếu Colab báo cần restart thì **Restart session** rồi chạy lại cell 1–2.

In [ ]:
import os, sys, shutil, json, subprocess
CODE_SOURCE   = 'git'     # 'git' | 'drive'
FUSHION_GIT   = 'https://github.com/chien24/Fushion_UEMR.git'
FUSHION_DRIVE = '/content/drive/MyDrive/uemr/code/Fushion'   # chỉ dùng khi CODE_SOURCE = 'drive'
FUSHION   = '/content/Fushion_UEMR'
OMNI_REPO = '/content/Omni-fix'

%cd /content
if CODE_SOURCE == 'git':
    if os.path.isdir(f'{FUSHION}/.git'):
        !git -C {FUSHION} pull -q
    else:
        shutil.rmtree(FUSHION, ignore_errors=True)
        !git clone -q {FUSHION_GIT} {FUSHION}
else:
    shutil.rmtree(FUSHION, ignore_errors=True)
    shutil.copytree(FUSHION_DRIVE, FUSHION,
                    ignore=shutil.ignore_patterns('__pycache__', '.pytest_cache', 'tmp', '*.npz'))
# repo có thể chứa omni_retrieval ở gốc hoặc trong thư mục con Fushion/
if not os.path.isdir(f'{FUSHION}/omni_retrieval') and os.path.isdir(f'{FUSHION}/Fushion/omni_retrieval'):
    FUSHION = f'{FUSHION}/Fushion'
assert os.path.isfile(f'{FUSHION}/omni_retrieval/search.py'), 'không thấy omni_retrieval trong ' + FUSHION
!git -C {FUSHION} log --oneline -1

if not os.path.isdir(OMNI_REPO):
    !git clone -q -b chien --single-branch https://github.com/im-xiaoming/Omni-fix.git {OMNI_REPO}
!git -C {OMNI_REPO} log --oneline -1

if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
os.environ['WAVE_HOME']  = '/content/WAVE_HOME'

if not os.path.exists('/content/.omni_deps_ok'):
    %cd {OMNI_REPO}
    !pip install -q -r requirements.txt
    !pip install -q -r training/requirements.txt
    !pip uninstall -y -q torchao
    !pip install -q -U torchao
    !pip install -q av decord pytest
    open('/content/.omni_deps_ok', 'w').close()
%cd {FUSHION}
import torch, transformers, peft, av, decord  # noqa: F401  (báo lỗi ở đây, không phải sau khi nạp model)
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__,
      '| gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '*** KHÔNG CÓ GPU ***')
if transformers.__version__ != '4.51.3':
    print('[!] transformers khác 4.51.3 (bản eval_youcookii đã chạy) -> sanity check ở cell 16 có thể lệch')

## 3. Tải WAVE-7B (khoảng 20 GB) và lấy checkpoint

`WAVE_HOME` trỏ vào thư mục **cha** của `WAVE-7B` (file BEATs nằm ở đó). Checkpoint `best` copy về `/content/my_checkpoint` như notebook inference (bỏ `global_step*/`). Kiểm tra đủ đường dẫn, kể cả metadata YouCook2 và file eval dùng cho sanity check.

In [ ]:
CKPT_SRC = '/content/drive/MyDrive/omniretriever/youcookii_ft_uemr/best'   # <-- ĐỔI nếu checkpoint ở chỗ khác
CKPT     = '/content/my_checkpoint'
BASE     = '/content/WAVE_HOME/WAVE-7B'
BEATS    = '/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt'
ORIG_ADAPTER = '/content/adapters/omniretriever-7b'   # adapter gốc YunzeLiu/OmniRetriever-7B (TAG='pt', ghép head)
RUN      = '/content/drive/MyDrive/omniretriever/youcookii_ft_uemr'
DRIVE_DATA = '/content/drive/MyDrive/Colab Notebooks/code KL/Omni/data/YouCookII'

if not os.path.exists(f'{BASE}/config.json'):
    !hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --quiet
if not os.path.exists(f'{CKPT}/adapter_model.safetensors'):
    os.makedirs(CKPT, exist_ok=True)
    for f in os.listdir(CKPT_SRC):
        if os.path.isfile(f'{CKPT_SRC}/{f}'):          # bỏ global_step*/ và thư mục con
            shutil.copy2(f'{CKPT_SRC}/{f}', f'{CKPT}/{f}')
!ls {CKPT}
if os.path.exists(f'{CKPT}/SOURCE.json'):
    print('checkpoint lấy từ:', json.load(open(f'{CKPT}/SOURCE.json')))

for p in [f'{BASE}/config.json', BEATS,
          f'{CKPT}/adapter_model.safetensors', f'{CKPT}/adapter_config.json',
          f'{DRIVE_DATA}/metadata/val_omni_video.jsonl',
          f'{DRIVE_DATA}/metadata/train_omni_video.dev.jsonl',
          f'{DRIVE_DATA}/metadata/train_omni_video.trainsplit.jsonl',
          f'{DRIVE_DATA}/videos',
          f'{RUN}/eval/best_val.json', f'{RUN}/eval/zeroshot_val.json',
          f'{OMNI_REPO}/scripts/eval_youcookii.py']:
    print('%-7s %s' % ('OK' if os.path.exists(p) else 'THIẾU', p))

## 4. Kiểm tra checkpoint có fusion head

Giống cell 6 của `Omni_inference_uemr.ipynb`: vector sinh ra từ `classify_linear` (cùng `beats_ln`, `beats_proj`, lưu qua `modules_to_save`). Thiếu chúng thì mọi thứ vẫn chạy nhưng vector vô nghĩa, không báo lỗi.

* Đủ head (trường hợp mong đợi: `checkpoint-1948` có `modules_to_save` và nặng ~790 MB) → đi tiếp.
* Thiếu head → ghép head từ adapter gốc vào `/content/my_checkpoint_full` và trỏ `CKPT` sang đó. Khi đó `eval_youcookii.py` lúc chọn checkpoint đã chạy **không** ghép, nên `best_val.json` không còn là mốc so sánh đúng: sanity check ở cell 16 chỉ in, không dừng.

In [ ]:
import gc
from safetensors.torch import load_file, save_file

HEADS  = ('classify_linear', 'beats_ln', 'beats_proj')
MERGED = '/content/my_checkpoint_full'
HEADS_MERGED = False

def inspect(path):
    sd    = load_file(f'{path}/adapter_model.safetensors')
    cfg_a = json.load(open(f'{path}/adapter_config.json'))
    heads = {k: v for k, v in sd.items() if 'lora_' not in k and any(h in k for h in HEADS)}
    found = sorted({h for h in HEADS if any(h in k for k in heads)})
    return sd, cfg_a, heads, found

sd, cfg_a, heads, found = inspect(CKPT)
print(f'[checkpoint] {CKPT}\n   head: {found} | tensor LoRA: {sum("lora_" in k for k in sd)} '
      f'| tổng tensor: {len(sd)} | modules_to_save: {cfg_a.get("modules_to_save")}')
missing = [h for h in HEADS if h not in found]

if missing:
    print(f'\n[!] Thiếu {missing} (LORA_ONLY=True không lưu head). Ghép head từ adapter gốc.')
    if not os.path.exists(f'{ORIG_ADAPTER}/adapter_model.safetensors'):
        os.system(f'hf download YunzeLiu/OmniRetriever-7B --local-dir {ORIG_ADAPTER} --quiet')
    sd_o, cfg_o, heads_o, found_o = inspect(ORIG_ADAPTER)
    assert not [h for h in HEADS if h not in found_o], f'Adapter gốc cũng thiếu head ({found_o})'
    assert {k for k in sd if 'lora_' in k} == {k for k in sd_o if 'lora_' in k}, 'Tên tensor LoRA lệch nhau'
    merged = dict(sd); merged.update(heads_o)
    os.makedirs(MERGED, exist_ok=True)
    save_file(merged, f'{MERGED}/adapter_model.safetensors')
    cfg_a['modules_to_save'] = cfg_o.get('modules_to_save')
    json.dump(cfg_a, open(f'{MERGED}/adapter_config.json', 'w'), indent=2)
    CKPT, HEADS_MERGED = MERGED, True
    del merged, sd_o, heads_o
    sd, cfg_a, heads, found = inspect(CKPT)
    print('[đã ghép] head:', found)

assert not [h for h in HEADS if h not in found], 'Vẫn thiếu head -> embedding sẽ là rác'
del sd, heads
gc.collect()
print(f'\nOK, CKPT = {CKPT}' + ('  (ĐÃ GHÉP HEAD: sanity check sẽ không dừng)' if HEADS_MERGED else ''))

## 5. Cấu hình

`TAG = 'ft'` trước; chạy lại với `TAG = 'pt'` cho cột (a) (adapter gốc, cùng manifest, khác thư mục cache). `EXTRA_PARTITIONS` thêm partition từ JSON `video_id → [[ts, te], ...]`, ví dụ `{'uniav_pred': '/content/drive/MyDrive/uemr/partitions/uniav_pred.json'}`.

Sanity check (cell 16) so với `best_val.json` (`ft`) hoặc `zeroshot_val.json` (`pt`) do `Omni_training_uemr.ipynb` ghi ra.

In [ ]:
from omni_retrieval.config import Config

TAG      = 'ft'        # 'ft' | 'pt'
SMOKE    = True        # True: 5 video encode thật; False: chạy đầy đủ
N_VIDEOS = 500         # tối đa 394 val + 104 dev = 498 (không dùng video train)
SEED     = 0
EXTRA_PARTITIONS = {}  # {'uniav_pred': '/content/drive/.../uniav_pred.json'}

def make_cfg(smoke):
    return Config.colab(tag=TAG, smoke=smoke, n_videos=N_VIDEOS, seed=SEED,
                        extra_partitions=EXTRA_PARTITIONS, omni_repo=OMNI_REPO,
                        drive_data=DRIVE_DATA, meta_dir=f'{DRIVE_DATA}/metadata', video_src=f'{DRIVE_DATA}/videos',
                        wave_path=BASE, beats_path=BEATS, adapter_ft=CKPT, adapter_pt=ORIG_ADAPTER,
                        ref_eval_json=f"{RUN}/eval/{'best_val' if TAG == 'ft' else 'zeroshot_val'}.json")

cfg = make_cfg(SMOKE)
if TAG == 'pt' and not os.path.exists(f'{ORIG_ADAPTER}/adapter_model.safetensors'):
    !hf download YunzeLiu/OmniRetriever-7B --local-dir {ORIG_ADAPTER} --quiet
ADAPTER = cfg.adapter
STRICT_SANITY = not SMOKE and not HEADS_MERGED
print('SMOKE    :', SMOKE, '| TAG:', TAG)
print('adapter  :', ADAPTER)
print('cache    :', cfg.tag_dir, '| output:', cfg.out_dir)
print('reference:', cfg.ref_eval_json, '(có)' if os.path.exists(cfg.ref_eval_json) else '(KHÔNG có)')

## 6. Unit test

Vector giả: MaxSim, timestamp `argmax`, gom theo video, metric trên ví dụ tính tay, collect/evaluate trên store giả. Không cần GPU. Phải pass hết.

In [ ]:
!cd {FUSHION} && python -m pytest omni_retrieval/tests -q -p no:cacheprovider

## 7. Dry-run (cấu hình đầy đủ, không ghi gì, không nạp model)

`subset` → `partitions` → `manifest` trên metadata trên Drive; duration đọc từ header mp4 trên Drive (8 luồng, chỉ header). In số video, số đoạn mỗi partition, số clip sau khi khử trùng lặp và 3 dòng manifest mẫu.

In [ ]:
from omni_retrieval.manifest import prepare
dry = prepare(make_cfg(smoke=False), write=False)
print('3 dòng segs_todo mẫu:')
for r in dry['segs_todo'][:3]:
    print(' ', json.dumps(r, ensure_ascii=False))
print('2 dòng caps_todo mẫu:')
for r in dry['caps_todo'][:2]:
    print(' ', json.dumps(r, ensure_ascii=False))

## 8. Test collect/evaluate với npz giả

Sinh `fake_av.npz` / `fake_text.npz` đúng format key `<id>__av` / `<id>__text` như CLI ghi ra, cho đúng bảng đoạn và caption của dry-run; vector text = vector clip GT của chính nó + nhiễu. Rồi chạy `collect` + `evaluate`: text→clip phải ≈ 100 % và `gt` phải thắng `global`.

In [ ]:
from omni_retrieval.fake import run_fake_pipeline
_ = run_fake_pipeline(dry, '/content/fake_test')

## 9. Chọn subset

Theo `SMOKE` ở cell 5. Query = caption val; gallery = mọi video val + video dev làm distractor. Lưu `common/subset_videos[_smoke].json`; chạy lại không probe lại.

In [ ]:
from omni_retrieval.subset import build_subset
subset = build_subset(cfg)
n_val = sum(r['is_query_source'] for r in subset)
print(f'SMOKE={SMOKE}: {len(subset)} video = {n_val} val (nguồn query) + {len(subset) - n_val} dev (distractor) -> {cfg.subset_path}')

## 10. Copy video về `/content/videos`

Đọc thẳng từ Drive rất chậm. `shutil.copy2` từng file, tối đa 8 luồng, không đọc nội dung vào RAM; bỏ qua file đã có đủ dung lượng.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
os.makedirs(cfg.video_root, exist_ok=True)

def copy_one(vid):
    src, dst = f'{cfg.video_src}/{vid}.mp4', f'{cfg.video_root}/{vid}.mp4'
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return 0
    shutil.copy2(src, dst + '.part')
    os.replace(dst + '.part', dst)
    return 1

with ThreadPoolExecutor(8) as pool:
    copied = sum(pool.map(copy_one, [r['video_id'] for r in subset]))
print(f'đã copy {copied}, có sẵn {len(subset) - copied}')
!df -h /content | tail -1

## 11. Sinh manifest

Bảng đoạn của mọi partition → khử trùng lặp theo `(video, ts, te)` làm tròn 0.01 s → `segs_todo` chỉ gồm đoạn **chưa có** trong cache, `caps_todo` chỉ gồm caption chưa encode.

In [ ]:
prepared = prepare(cfg)
print(cfg.segs_todo)
print(cfg.caps_todo)

## 12. Ước lượng thời gian

Encode 20 clip (giữ lại trong cache, không phí), đo giây/clip, nhân với số clip còn lại.

In [ ]:
ENC = (f'cd {FUSHION} && python -m omni_retrieval.encode --omni-repo {OMNI_REPO} '
       f'--base-model {BASE} --beats-path {BEATS} --adapter {ADAPTER} --video-root {cfg.video_root}')
timing = cfg.tag_dir / 'timing_20.json'
n_left = len(prepared['segs_todo'])
if n_left >= 20 and not timing.exists():
    !{ENC} --modality av --manifest {cfg.segs_todo} --store {cfg.av_store} --limit 20 --timing-out {timing}
if timing.exists():
    t = json.load(open(timing))
    spc = t['sec_per_clip']
    print(f"giây/clip (median): {spc:.2f} | nạp model: {t['model_load_sec']:.0f} s | lỗi: {t['failed']}")
    print(f'còn {n_left} clip -> khoảng {n_left * spc / 3600:.2f} giờ GPU (+ nạp model)')

## 13. Encode `av`

Batch 1 như eval (fusion head lấy token cuối cố định, padding làm đổi embedding); 4 worker decode song song với GPU. Ghi thẳng lên Drive theo chunk 200 clip. Phiên Colab ngắt: chạy lại cell 1–5, 11 rồi cell này, chỉ encode phần còn thiếu. Nếu worker bị kill vì RAM, đổi `--num-workers 0`.

In [ ]:
!{ENC} --modality av --manifest {cfg.segs_todo} --store {cfg.av_store} --num-workers 4
failed = cfg.av_store / 'failed.jsonl'
if failed.exists():
    print('CÓ CLIP LỖI, xem', failed)
    print(open(failed).read()[:2000])

## 14. Encode `text`

Caption val của subset, cùng nhánh text của eval (batch theo độ dài token, không padding).

In [ ]:
!{ENC} --modality text --manifest {cfg.caps_todo} --store {cfg.text_store} --batch-size 32

## 15. Gom cache

`segments.npz`, `text.npz`, `meta.json` trong `omni_cache/<tag>/` (smoke: `<tag>/smoke/`). Đọc store từng key, ghi fp16. Báo lỗi nếu thiếu embedding.

In [ ]:
from omni_retrieval.collect import collect
prepared = prepare(cfg, verbose=False)   # làm mới bảng sau khi encode
meta = collect(cfg, prepared)

## 16. Đánh giá

**Sanity check bắt buộc:** text→clip trên đoạn `gt` (gallery giống eval) phải khớp t2m R@1 của eval; lệch > 1 điểm thì in cảnh báo lớn và dừng (smoke hoặc đã ghép head: chỉ in). Sau đó bảng text→video theo partition, lưu `results_<tag>.csv`.

**Smoke xong và ổn → sửa `SMOKE = False` ở cell 5, chạy lại cell 5 rồi 9–16.**

In [ ]:
from IPython.display import Markdown, display
from omni_retrieval.evaluate import evaluate, markdown, sanity_check, save_csv

sanity = sanity_check(cfg.out_dir, cfg.ref_eval_json, tol=1.0, strict=STRICT_SANITY)
parts = ['global', 'event_single', 'uni_M_gt', 'gt'] + list(EXTRA_PARTITIONS)
rows = evaluate(cfg.out_dir, parts, agg='max')
display(Markdown(markdown(rows)))
save_csv(rows, cfg.out_dir / f'results_{TAG}.csv')
json.dump({'sanity': sanity, 'rows': rows}, open(cfg.out_dir / f'results_{TAG}.json', 'w'), indent=1, default=float)
print('->', cfg.out_dir / f'results_{TAG}.csv')

## 17. Giải phóng bộ nhớ trước khi nạp model

Encode chạy trong tiến trình con nên kernel chưa giữ model; cell này xoá các bảng lớn còn lại và dọn cache CUDA. Model đã nạp ở cell 18 thì **giữ lại** (không nạp lần hai), trừ khi `UNLOAD_MODEL = True`.

In [ ]:
UNLOAD_MODEL = False
for name in ('dry', 'prepared', 'rows', 'meta', 'sanity', 'index', 'text'):
    globals().pop(name, None)
if UNLOAD_MODEL and 'MODEL' in globals():
    del MODEL, PROC
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU đang dùng: {torch.cuda.memory_allocated() / 2**30:.1f} GiB')
print('model trong kernel:', 'có (sẽ dùng lại)' if 'MODEL' in globals() else 'chưa')

## 18. Query tương tác

* `LOAD_MODEL = False`: không cần GPU, lấy caption val thứ `CAPTION_INDEX` trong `text.npz` làm query.
* `LOAD_MODEL = True`: nạp model **một lần** (cùng checkpoint với cache, cùng nhánh text của eval) rồi gõ query; Enter trống để thoát. Lần nạp đầu tự kiểm tra: encode lại một caption trong cache, cosine phải ≈ 1.

Kết quả: `video_id`, điểm, `[t_s, t_e]` của đoạn khớp nhất, caption GT gần đoạn đó nhất, và clip cắt bằng ffmpeg.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Video, display
from omni_retrieval.search import SegmentIndex, tiou
from omni_retrieval.evaluate import load_text

PARTITION     = 'gt'   # 'global' | 'gt' | 'uni_M_gt' | 'event_single' | partition thêm
TOP_K         = 5
SHOW_CLIPS    = 2      # số clip đầu bảng nhúng vào để xem
LOAD_MODEL    = False
CAPTION_INDEX = 0

index = SegmentIndex.load(cfg.out_dir, PARTITION)
text  = load_text(cfg.out_dir)

def nearest_gt_caption(vid, seg):
    sel = np.where(text['video_id'] == vid)[0]
    if not len(sel):
        return '(video dev, không có caption query)'
    b = max(sel, key=lambda i: tiou(seg, (text['gt_ts'][i], text['gt_te'][i])))
    return f"{text['sentence'][b]}  [{text['gt_ts'][b]:.0f}-{text['gt_te'][b]:.0f}]"

def cut_clip(vid, ts, te):
    os.makedirs('/content/clips', exist_ok=True)
    out = f'/content/clips/{vid}_{ts:.1f}_{te:.1f}.mp4'
    if not os.path.exists(out):
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', str(ts), '-to', str(te),
                        '-i', f'{cfg.video_root}/{vid}.mp4', '-vf', 'scale=-2:240',
                        '-c:v', 'libx264', '-preset', 'veryfast', '-c:a', 'aac', out], check=True)
    return out

def show(q_vec, query, gt_video=None):
    hits = index.search(q_vec, top_k=TOP_K)
    print(f'[{PARTITION}] "{query}"' + (f'   (video GT: {gt_video})' if gt_video else ''))
    display(pd.DataFrame([{'hạng': i + 1, 'video_id': h['video_id'], 'điểm': round(h['score'], 4),
                           '[t_s, t_e]': f"[{h['best_segment'][0]:.1f}, {h['best_segment'][1]:.1f}]",
                           'caption GT gần nhất': nearest_gt_caption(h['video_id'], h['best_segment']),
                           'đúng video': 'x' if h['video_id'] == gt_video else ''}
                          for i, h in enumerate(hits)]))
    for h in hits[:SHOW_CLIPS]:
        display(Video(cut_clip(h['video_id'], *h['best_segment']), embed=True, width=420))

if not LOAD_MODEL:
    i = CAPTION_INDEX
    show(text['emb'][i], text['sentence'][i], gt_video=text['video_id'][i])
else:
    from omni_retrieval.encode import encode_texts, load_model
    if 'MODEL' not in globals():   # không bao giờ nạp model lần hai trong một phiên
        MODEL, PROC, _ = load_model(OMNI_REPO, BASE, BEATS, ADAPTER)
        chk = encode_texts(MODEL, PROC.tokenizer, [str(text['sentence'][0])])[0]
        print('kiểm tra nhánh text: cos(query mới, cache) =', round(float(chk @ text['emb'][0]), 4))
    while True:
        q = input('query (Enter để thoát): ').strip()
        if not q:
            break
        show(encode_texts(MODEL, PROC.tokenizer, [q])[0], q)